# O Problema
Sliding Puzzle - Bloco Deslizante

In [1]:
# !wget -qq https://miro.medium.com/max/700/1*W7jg4GmEjGBypd9WPktasQ.gif
from IPython.display import Image
Image(url='https://miro.medium.com/max/700/1*W7jg4GmEjGBypd9WPktasQ.gif',width=200)

# Resolver o quebra-cabeças usando Buscas

## Busca em largura

In [4]:
from collections import deque
import random

class PuzzleState:
    GOAL_BOARD = [[1, 2, 3], [4, 5, 6], [7, 8, 0]]

    def __init__(self, board, parent=None, move=None):
        self.board = board
        self.parent = parent
        self.move = move
        self.zero_pos = self._find_zero()

    def _find_zero(self):
        for r in range(3):
            for c in range(3):
                if self.board[r][c] == 0:
                    return r, c

    def is_goal(self):
        return self.board == self.GOAL_BOARD

    def get_neighbors(self):
        neighbors = []
        zr, zc = self.zero_pos

        # Define possible moves: (dr, dc, move_name)
        moves = [
            (-1, 0, "U"),  # Up
            (1, 0, "D"),   # Down
            (0, -1, "L"),  # Left
            (0, 1, "R")    # Right
        ]

        for dr, dc, move_name in moves:
            nr, nc = zr + dr, zc + dc

            if 0 <= nr < 3 and 0 <= nc < 3:
                new_board = [row[:] for row in self.board] # Deep copy
                new_board[zr][zc], new_board[nr][nc] = new_board[nr][nc], new_board[zr][zc]
                neighbors.append(PuzzleState(new_board, self, move_name))
        return neighbors

    def __str__(self):
        return "\n".join([" ".join(map(str, row)) for row in self.board])

    def __eq__(self, other):
        return self.board == other.board

    def __hash__(self):
        return hash(str(self.board))

def generate_random_puzzle():
    """Gera um 8-puzzle válido (solucionável)"""
    while True:
        numbers = [1, 2, 3, 4, 5, 6, 7, 8, 0]
        random.shuffle(numbers)
        board = [numbers[i*3:(i+1)*3] for i in range(3)]

        # Verifica se é solucionável
        flat = [tile for row in board for tile in row if tile != 0]
        inversions = sum(1 for i in range(len(flat)) for j in range(i+1, len(flat)) if flat[i] > flat[j])
        if inversions % 2 == 0:
            return board

def bfs(start):
    start_state = PuzzleState(start)
    if start_state.is_goal():
        return []

    queue = deque([start_state])
    visited = set([str(start_state.board)])

    while queue:
        current = queue.popleft()

        for neighbor in current.get_neighbors():
            if neighbor.is_goal():
                path = []
                while neighbor.parent:
                    path.append(neighbor.move)
                    neighbor = neighbor.parent
                return path[::-1]

            if str(neighbor.board) not in visited:
                visited.add(str(neighbor.board))
                queue.append(neighbor)

    return None

# ESCOLHA: usar tabuleiro fixo OU aleatório
use_random = True  # Mude para False para usar o fixo

if use_random:
    print("Gerando tabuleiro aleatório...")
    start_board = generate_random_puzzle()
else:
    # Tabuleiro fixo (seu exemplo)
    start_board = [
        [1, 3, 5],
        [4, 2, 6],
        [7, 0, 8]
    ]

print("Tabuleiro inicial:")
for row in start_board:
    print(' '.join(str(x) if x != 0 else '_' for x in row))

solution = bfs(start_board)
print("\nSolução encontrada com BFS:", solution)
print("Número de movimentos:", len(solution) if solution else 0)

if solution:
    print("\nSequência de movimentos:")
    move_names = {"U": "↑ Cima", "D": "↓ Baixo", "L": "← Esquerda", "R": "→ Direita"}
    for i, move in enumerate(solution, 1):
        print(f"{i}. {move_names[move]}")
else:
    print("Não foi possível resolver este tabuleiro.")

Gerando tabuleiro aleatório...
Tabuleiro inicial:
1 5 3
_ 2 8
6 4 7

Solução encontrada com BFS: ['D', 'R', 'R', 'U', 'U', 'L', 'D', 'L', 'U', 'R', 'R', 'D', 'L', 'U', 'L', 'D', 'D', 'R', 'R']
Número de movimentos: 19

Sequência de movimentos:
1. ↓ Baixo
2. → Direita
3. → Direita
4. ↑ Cima
5. ↑ Cima
6. ← Esquerda
7. ↓ Baixo
8. ← Esquerda
9. ↑ Cima
10. → Direita
11. → Direita
12. ↓ Baixo
13. ← Esquerda
14. ↑ Cima
15. ← Esquerda
16. ↓ Baixo
17. ↓ Baixo
18. → Direita
19. → Direita


## Busca em Profundidade

In [5]:
from collections import deque
import random

class PuzzleState:
    GOAL_BOARD = [[1, 2, 3], [4, 5, 6], [7, 8, 0]]

    def __init__(self, board, parent=None, move=None):
        self.board = board
        self.parent = parent
        self.move = move
        self.zero_pos = self._find_zero()

    def _find_zero(self):
        for r in range(3):
            for c in range(3):
                if self.board[r][c] == 0:
                    return r, c

    def is_goal(self):
        return self.board == self.GOAL_BOARD

    def get_neighbors(self):
        neighbors = []
        zr, zc = self.zero_pos

        moves = [
            (-1, 0, "U"),  # Cima
            (1, 0, "D"),   # Baixo
            (0, -1, "L"),  # Esquerda
            (0, 1, "R")    # Direita
        ]

        for dr, dc, move_name in moves:
            nr, nc = zr + dr, zc + dc
            if 0 <= nr < 3 and 0 <= nc < 3:
                new_board = [row[:] for row in self.board]
                new_board[zr][zc], new_board[nr][nc] = new_board[nr][nc], new_board[zr][zc]
                neighbors.append(PuzzleState(new_board, self, move_name))
        return neighbors

    def __eq__(self, other):
        return self.board == other.board

    def __hash__(self):
        return hash(str(self.board))


def generate_random_puzzle():
    while True:
        numbers = [1, 2, 3, 4, 5, 6, 7, 8, 0]
        random.shuffle(numbers)
        board = [numbers[i*3:(i+1)*3] for i in range(3)]

        flat = [tile for row in board for tile in row if tile != 0]
        inversions = sum(
            1 for i in range(len(flat))
            for j in range(i+1, len(flat))
            if flat[i] > flat[j]
        )

        if inversions % 2 == 0:
            return board


def dfs(start, max_depth=50):
    start_state = PuzzleState(start)
    stack = [start_state]
    visited = set([str(start_state.board)])

    while stack:
        current = stack.pop()

        if current.is_goal():
            path = []
            while current.parent:
                path.append(current.move)
                current = current.parent
            return path[::-1]

        if len(get_path(current)) >= max_depth:
            continue

        for neighbor in current.get_neighbors():
            if str(neighbor.board) not in visited:
                visited.add(str(neighbor.board))
                stack.append(neighbor)

    return None


def get_path(state):
    path = []
    while state.parent:
        path.append(state.move)
        state = state.parent
    return path


# ESCOLHA
use_random = True

if use_random:
    print("Gerando tabuleiro aleatório...")
    start_board = generate_random_puzzle()
else:
    start_board = [
        [1, 3, 5],
        [4, 2, 6],
        [7, 0, 8]
    ]

print("Tabuleiro inicial:")
for row in start_board:
    print(' '.join(str(x) if x != 0 else '_' for x in row))

solution = dfs(start_board, max_depth=50)

print("\nSolução encontrada com DFS:", solution)
print("Número de movimentos:", len(solution) if solution else 0)

if solution:
    print("\nSequência de movimentos:")
    move_names = {"U": "↑ Cima", "D": "↓ Baixo", "L": "← Esquerda", "R": "→ Direita"}
    for i, move in enumerate(solution, 1):
        print(f"{i}. {move_names[move]}")
else:
    print("Não foi possível resolver dentro do limite de profundidade.")


Gerando tabuleiro aleatório...
Tabuleiro inicial:
_ 6 8
5 4 7
1 3 2

Solução encontrada com DFS: None
Número de movimentos: 0
Não foi possível resolver dentro do limite de profundidade.


## Discorra sobre o desempenho dos métodos em questões de:


1.   Consumo de memória
2.   Processamento

